# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SodiqAbdulwaris/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane:** Content Refresh Prioritization (freestyle, rooted in the shipped pipeline).

**Research question:** which of a client's existing pages should an SEO/content editor review first? Editors have limited hours; the starter data shows 54.2% of pages are "declining," and the obvious signal -- keyword search volume -- barely correlates with actual traffic (`corr(search_volume, impressions_90d) = 0.001`, ML-02).

**Decision this supports:** ordering a weekly review backlog so editors open the highest-value page first. **Who acts:** the SEO/content editor, who reworks a flagged page (refresh, expand, or fix CTR). **Cost of a wrong call:** a false flag wastes an editor's hour on a healthy page; a missed decline lets a traffic-losing page sit untouched -- so the method must favor recall at the top of the queue without drowning editors in false alarms (ML-02).

**Task type:** ranking/scoring, not classification -- the deliverable is a priority-ordered queue, not a yes/no verdict (ML-03).

In [1]:
import sys
sys.path.insert(0, "scripts")
import json
from pathlib import Path

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows, cols:", df.shape)
print("declining rate (label proxy):", round((df['trend_direction'] == 'down').mean(), 3))
print("corr(search_volume, impressions_90d):", round(df["search_volume"].corr(df["impressions_90d"]), 3))


rows, cols: (30000, 44)
declining rate (label proxy): 0.542
corr(search_volume, impressions_90d): 0.001


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release used:** the bundled starter export, `data/raw/content_refresh_anonymized.csv` -- 30,000 rows, one row per pseudonymized content item, 32 pseudonymized clients, trailing 90-day metrics as of one export date (ML-04). The full FlyRank warehouse (519,606 items, 104 clients) exists but is out of scope for this capstone; numbers here describe this teaching slice, not the whole portfolio.

**Excluded, with why (ML-04's Feature/Label/Context/Excluded contract):**
- `trend_direction`, `trend_pct` -- the label's source; `is_declining_label` is defined as `trend_direction == "down"`.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` -- proven (below) to be `trend_pct`'s literal inputs, not just correlated with it.
- `provider_used`, `model_used` -- content-generation metadata, not a performance signal.
- raw `impressions_90d`/`clicks_90d`/`sessions_90d`/`ai_sessions_90d` -- replaced by their `log1p` versions (heavy-tailed).
- `content_id`, `client_id` -- context only, used for grouping and the client-holdout split, never as features.

No client names, URLs, domains, or raw queries exist anywhere in this dataset or this paper.

In [2]:
# Grain + leakage proof, carried over from ML-04/ML-07
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("duplicate content_id (grain check):", df["content_id"].duplicated().sum())
print("distinct client_id:", df["client_id"].nunique())

recomputed = ((df["impressions_last_30d"] - df["impressions_prev_30d"])
              / df["impressions_prev_30d"].replace(0, np.nan) * 100)
comparable = df["trend_pct"].notna() & recomputed.notna()
match = np.isclose(recomputed[comparable], df.loc[comparable, "trend_pct"], atol=0.05)
print("trend_pct recomputed-from-raw-columns match rate:", round(match.mean(), 3),
      "-- confirms last_30d/prev_30d ARE the label's inputs, not just correlated")


duplicate content_id (grain check): 0
distinct client_id: 32
trend_pct recomputed-from-raw-columns match rate: 1.0 -- confirms last_30d/prev_30d ARE the label's inputs, not just correlated


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Baseline (ML-07):** a transparent rule, frozen once model work started -- a page is worth reviewing if it's stale (`days_since_last_update >= 180`) AND still visible (`impressions_90d >= 500`), ranked by exposure. Only 17 of 30,000 pages ever pass both conditions (94% of those 17 are correctly declining), so everything past rank 17 is an arbitrary zero-score tie. On the held-out clients only 3 pages score above zero; tie-aware Precision@K uses the tied group's expected hit rate for the remaining slots instead of letting row order or pandas choose the reported baseline.

**Method (ML-08):** Logistic Regression, then Random Forest -- the toolkit's guidance for an observed-label ranking task is to start readable and add complexity only if it earns its keep. Features are exactly ML-04's **Feature** bucket, plus `has_*` missingness flags (search_volume/word_count/scroll_rate go blank by `content_type`, ML-04 -- a blind `fillna(0)` would silently encode content type into the model) and `log1p` of the raw traffic totals.

**Label:** `is_declining_label = (trend_direction == "down")` -- an observed proxy from one 90-day snapshot's 30-vs-30-day impression comparison, not a verified future outcome.

**Validation design:** `GroupShuffleSplit(test_size=0.25, random_state=42)` grouped on `client_id` -- 24 clients train / 8 clients test, so no client's pages appear on both sides. ML-09 quantified why this matters: the same model scored on a random row-level split instead reads 16 points higher at Precision@50 (0.86 vs 0.70), because 31 of 32 clients leak across train/test under a random split -- the grouped number is the one this paper reports.

**Leakage check (ML-09):** deliberately added `trend_pct` (the label's excluded source column) back into the honest feature set as an attack test -- Precision@50 jumped from 0.70 to 1.00 and ROC AUC to 0.999, confirming the test harness actually detects a real leak rather than just never finding one, and confirming the reported feature set is clean of it.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k, simple_svg_bar_chart

# has_* flags before filling -- missingness tracks content_type (ML-04)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_scroll_data"] = df["scroll_rate"].notna().astype(int)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

numeric_features = list(MODEL_NUMERIC_FEATURES) + ["has_keyword_data", "has_word_count", "has_scroll_data"]
for col in numeric_features:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_features = list(MODEL_CATEGORICAL_FEATURES)
for col in categorical_features:
    df[col] = df[col].fillna("unknown").astype(str)

X = pd.concat([df[numeric_features], pd.get_dummies(df[categorical_features], prefix=categorical_features)], axis=1)
y = df["is_declining_label"]

# baseline (ML-07, frozen)
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
print("train pages:", len(train_idx), "| test pages:", len(test_idx),
      "| test clients:", df.iloc[test_idx]["client_id"].nunique())

# leakage attack: trend_pct pushed back in on the SAME split, then discarded
df["trend_pct_filled"] = df["trend_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)
X_attack = X.copy()
X_attack["trend_pct"] = df["trend_pct_filled"]
attack_numeric = numeric_features + ["trend_pct"]
Xa_train, Xa_test = X_attack.iloc[train_idx].copy(), X_attack.iloc[test_idx].copy()
scaler_a = StandardScaler()
Xa_train[attack_numeric] = scaler_a.fit_transform(Xa_train[attack_numeric])
Xa_test[attack_numeric] = scaler_a.transform(Xa_test[attack_numeric])
attack_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
attack_model.fit(Xa_train, y.iloc[train_idx])
p_attack = attack_model.predict_proba(Xa_test)[:, 1]
print("leakage attack (trend_pct added back in) precision@50:",
      round(precision_at_k(y.iloc[test_idx], p_attack, 50), 3), "-- confirms the harness detects a real leak")


train pages: 22885 | test pages: 7115 | test clients: 8


leakage attack (trend_pct added back in) precision@50: 1.0 -- confirms the harness detects a real leak


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Logistic Regression wins both the baseline and Random Forest at Precision@20 (0.80) and Precision@50 (0.70) on the client-holdout test set (base rate 0.517) -- report that as the finding rather than defaulting to the fancier model.

**Error pattern (ML-08):** false positives cluster on moderate-traffic pages actually trending up or stable -- they look like decliners on every pre-decision signal but the direction was wrong, since trend history itself is excluded as leakage. False negatives cluster on very-low-traffic old pages that ARE declining but carry almost no signal to work with. Top features (`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`) are all plausible and none dominates suspiciously -- the sanity check that matters, since a single feature owning nearly all the importance would suggest leakage, and that isn't what this shows.

In [4]:
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
logreg.fit(X_train, y_train)
p_log = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5,
                             class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]

baseline_test_scores = df.iloc[test_idx]["baseline_score"]
base_rate = float(y_test.mean())

comparison = pd.DataFrame({
    "model": ["baseline_rules", "logistic_regression", "random_forest"],
    "roc_auc": [None, roc_auc_score(y_test, p_log), roc_auc_score(y_test, p_rf)],
    "avg_precision": [None, average_precision_score(y_test, p_log), average_precision_score(y_test, p_rf)],
    "precision_at_20": [precision_at_k(y_test, baseline_test_scores, 20), precision_at_k(y_test, p_log, 20), precision_at_k(y_test, p_rf, 20)],
    "precision_at_50": [precision_at_k(y_test, baseline_test_scores, 50), precision_at_k(y_test, p_log, 50), precision_at_k(y_test, p_rf, 50)],
}).round(3)
print(f"base rate (test, {len(y_test)} pages, {df.iloc[test_idx]['client_id'].nunique()} clients): {base_rate:.3f}\n")
print(comparison.to_string(index=False))

# honest-split disclosure (ML-09): the same model under a random, non-grouped split
rand_train_idx, rand_test_idx = train_test_split(np.arange(len(df)), test_size=0.25, random_state=42, stratify=y)
Xr_train, Xr_test = X.iloc[rand_train_idx].copy(), X.iloc[rand_test_idx].copy()
scaler_r = StandardScaler()
Xr_train[numeric_features] = scaler_r.fit_transform(Xr_train[numeric_features])
Xr_test[numeric_features] = scaler_r.transform(Xr_test[numeric_features])
logreg_r = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
logreg_r.fit(Xr_train, y.iloc[rand_train_idx])
p_log_r = logreg_r.predict_proba(Xr_test)[:, 1]
random_split_p50 = round(precision_at_k(y.iloc[rand_test_idx], p_log_r, 50), 3)
shared_clients = len(set(df.iloc[rand_train_idx]["client_id"]) & set(df.iloc[rand_test_idx]["client_id"]))
print(f"\nDISCLOSED: same model, random row-level split -- precision@50 {random_split_p50}",
      f"({shared_clients}/{df['client_id'].nunique()} clients leak across train/test).",
      "The grouped number above (0.70) is the one this paper reports as honest.")

# feature importances / coefficients, for the interpretation this section leans on
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\ntop 5 random forest feature importances:\n", rf_importance.head(5).round(3))

Path("docs/assets").mkdir(parents=True, exist_ok=True)
simple_svg_bar_chart(
    "Precision@50 (client-holdout): baseline vs model vs base rate",
    ["base rate", "baseline rule", "logistic regression", "random forest"],
    [base_rate, comparison.loc[0, "precision_at_50"], comparison.loc[1, "precision_at_50"], comparison.loc[2, "precision_at_50"]],
    Path("docs/assets/precision_comparison.svg"),
)
simple_svg_bar_chart(
    "Precision@50: grouped (honest) vs random split",
    ["grouped client-holdout", "random row-level split"],
    [comparison.loc[1, "precision_at_50"], random_split_p50],
    Path("docs/assets/split_honesty.svg"),
)
print("\nwrote docs/assets/precision_comparison.svg and split_honesty.svg")


base rate (test, 7115 pages, 8 clients): 0.517

              model  roc_auc  avg_precision  precision_at_20  precision_at_50
     baseline_rules      NaN            NaN            0.589            0.545
logistic_regression    0.610          0.605            0.800            0.700
      random_forest    0.603          0.587            0.550            0.560



DISCLOSED: same model, random row-level split -- precision@50 0.86 (31/32 clients leak across train/test). The grouped number above (0.70) is the one this paper reports as honest.

top 5 random forest feature importances:
 days_with_impressions    0.149
log_impressions_90d      0.130
avg_position             0.103
content_age_days         0.087
char_count               0.046
dtype: float64

wrote docs/assets/precision_comparison.svg and split_honesty.svg


## 5. Limitations

*What this work cannot claim.*

- **No real future outcome.** `is_declining_label` is a proxy from a 30-vs-30-day comparison inside one 90-day snapshot, not an observed outcome after anyone acted. Nothing here shows that refreshing a flagged page actually recovers it.
- **A sample, not the population.** 30,000 pages / 32 clients here vs. the documented full warehouse of 519,606 items / 104 clients -- rates measured on this slice (the 54.2% decline rate, the 0.70 Precision@50) describe this teaching sample, not every client.
- **Client-specific memorization.** The 0.86-vs-0.70 random-vs-grouped gap (Section 3/4) shows part of what the model knows is patterns specific to the 24 training clients -- a genuinely new client may score below the 0.70 headline.
- **Feedback loop.** Once editors act on this queue, refreshed pages stop being an untouched observational sample. A future retrain on post-intervention data risks learning the review policy instead of the content -- a risk to monitor, not something this analysis can rule out on its own.
- **Missingness tracks content_type, not chance.** Any read leaning on `search_volume`/`word_count` inherits a content-type mix bias unless controlled for.
- **No causal claims.** No experiment, no randomization, no control group. Everything here is observed / directional / decision-support -- never "refreshing this page will improve traffic."

In [5]:
# No new computation -- limitations are qualitative, evidenced by the numbers already
# produced in Sections 2-4 above (grain checks, the 0.86-vs-0.70 split gap, missingness rates).
print("declining rate, this sample:", round(df['is_declining_label'].mean(), 3))
print("this sample: 30,000 pages / 32 clients")
print("documented full warehouse: 519,606 items / 104 clients (docs/data-dictionary.md)")


declining rate, this sample: 0.542
this sample: 30,000 pages / 32 clients
documented full warehouse: 519,606 items / 104 clients (docs/data-dictionary.md)


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The ML-08 winner (Logistic Regression), refit on all 30,000 pages, tiered by its own probability into high/medium/low confidence. Reason codes reuse the baseline's human-checkable flags where they fire, plus `model_flagged` when a high-tier page passes none of them -- 75.8% of high-tier pages -- an honest label for "the model's combined read of many weak signals says review this, but no single rule explains why," which is exactly the model's edge over the rule baseline, and exactly why a human closes the loop, not a machine.

**What a person must check before acting:** the page isn't already scheduled for retirement/consolidation; for `model_flagged` rows specifically, read the page since no simple rule explains the flag; the export is recent. **Never automate:** auto-publishing, auto-redirecting, or auto-deleting from this queue; exposing the model's probability to a client as a verdict; judging a content creator's performance from this label.

In [6]:
X_deploy = X.copy()
scaler_d = StandardScaler()
X_deploy[numeric_features] = scaler_d.fit_transform(X_deploy[numeric_features])
deploy_model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
deploy_model.fit(X_deploy, y)
df["model_probability"] = deploy_model.predict_proba(X_deploy)[:, 1]

def confidence_tier(p):
    if p >= 0.70:
        return "high"
    if p >= 0.50:
        return "medium"
    return "low"

df["confidence_tier"] = df["model_probability"].apply(confidence_tier)
tier_counts = df["confidence_tier"].value_counts()
print("confidence tier counts:\n", tier_counts, sep="")

ctr_median = df["ctr"].median()

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_but_visible")
    if row["word_count"] > 0 and row["word_count"] < 1200:
        reasons.append("thin_content")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["ctr"] < ctr_median:
        reasons.append("page_one_low_ctr")
    if not reasons and row["confidence_tier"] == "high":
        reasons.append("model_flagged")
    if not reasons:
        reasons.append("no_flag")
    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)
high_tier = df[df["confidence_tier"] == "high"]
model_flagged_share = round(float((high_tier["reason_codes"] == "model_flagged").mean()), 3)
print("\nshare of high-tier pages with no simple rule explanation (model_flagged):", model_flagged_share)

simple_svg_bar_chart(
    "Action queue by confidence tier (30,000 pages)",
    ["high (review priority)", "medium (monitor)", "low (no action)"],
    [tier_counts.get("high", 0), tier_counts.get("medium", 0), tier_counts.get("low", 0)],
    Path("docs/assets/tier_distribution.svg"),
)
print("wrote docs/assets/tier_distribution.svg")


confidence tier counts:
confidence_tier
low       13639
medium    11242
high       5119
Name: count, dtype: int64



share of high-tier pages with no simple rule explanation (model_flagged): 0.758
wrote docs/assets/tier_distribution.svg


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three charts (Sections 4 and 6 above) plus one committed metrics file the deployed page's numbers trace back to -- the same pattern as ML-07/08/09/10's `work/outputs/*.json` receipts. "Three great charts beat ten fillers": precision comparison, the honest-split disclosure, and the action-queue tier breakdown are the three that carry this paper's argument.

In [7]:
results_dict = comparison.set_index("model").to_dict(orient="index")
for row in results_dict.values():
    for key, value in row.items():
        if isinstance(value, float) and np.isnan(value):
            row[key] = None

summary = {
    "sample": {"rows": int(len(df)), "clients": int(df["client_id"].nunique()), "declining_rate": round(float(df["is_declining_label"].mean()), 3)},
    "split": {"test_pages": int(len(y_test)), "test_clients": int(df.iloc[test_idx]["client_id"].nunique()), "base_rate_test": round(base_rate, 3)},
    "results": results_dict,
    "random_split_precision_at_50": random_split_p50,
    "leakage_attack_precision_at_50": round(precision_at_k(y.iloc[test_idx], p_attack, 50), 3),
    "action_queue_tier_counts": tier_counts.to_dict(),
    "high_tier_model_flagged_share": model_flagged_share,
    "top_rf_features": rf_importance.head(5).round(3).to_dict(),
}

Path("work/outputs").mkdir(parents=True, exist_ok=True)
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Artifacts written for the deployed paper:")
print(" - docs/assets/precision_comparison.svg")
print(" - docs/assets/split_honesty.svg")
print(" - docs/assets/tier_distribution.svg")
print(" - work/outputs/capstone_metrics.json (committed -- the receipts these numbers trace back to)")
print()
print(json.dumps(summary, indent=2))


Artifacts written for the deployed paper:
 - docs/assets/precision_comparison.svg
 - docs/assets/split_honesty.svg
 - docs/assets/tier_distribution.svg
 - work/outputs/capstone_metrics.json (committed -- the receipts these numbers trace back to)

{
  "sample": {
    "rows": 30000,
    "clients": 32,
    "declining_rate": 0.542
  },
  "split": {
    "test_pages": 7115,
    "test_clients": 8,
    "base_rate_test": 0.517
  },
  "results": {
    "baseline_rules": {
      "roc_auc": null,
      "avg_precision": null,
      "precision_at_20": 0.589,
      "precision_at_50": 0.545
    },
    "logistic_regression": {
      "roc_auc": 0.61,
      "avg_precision": 0.605,
      "precision_at_20": 0.8,
      "precision_at_50": 0.7
    },
    "random_forest": {
      "roc_auc": 0.603,
      "avg_precision": 0.587,
      "precision_at_20": 0.55,
      "precision_at_50": 0.56
    }
  },
  "random_split_precision_at_50": 0.86,
  "leakage_attack_precision_at_50": 1.0,
  "action_queue_tier_counts": {
  

## 8. ML-12 — 5-Minute Demo, Social Cut, Employer Summary

### 5-minute demo outline

| Time | Beat | Say / show |
|---|---|---|
| 0:00–0:30 | Hook | "54.2% of pages in this portfolio are declining, and the obvious signal — search volume — has a 0.001 correlation with actual traffic. Volume tells you almost nothing about which page needs attention." |
| 0:30–1:15 | The naive fix fails | Show the baseline rule (stale + visible). Say it out loud in one sentence. Then: "It only ever fires on 17 of 30,000 pages — everything past rank 17 is a coin flip, not a recommendation." |
| 1:15–2:45 | The honest result | Show `precision_comparison.svg`. "A Logistic Regression model, trained on pre-decision signals only, hits Precision@50 of 0.70 against a 0.517 base rate and a 0.545 tie-aware baseline — measured on 8 clients it never trained on." Then show `split_honesty.svg` and name the trap avoided: "score it on a random split instead and it reads 0.86 — 16 points of pure memorization. That's not the number we report." |
| 2:45–3:45 | The output | Show `tier_distribution.svg` and one row of the action queue. "5,119 pages land in the review-priority tier. 76% of those carry no simple rule explanation — that's the model doing something a human rule can't, which is exactly why a human still has to read the page before acting on it." |
| 3:45–4:30 | Limits, said before anyone asks | "This is decision-support on one snapshot, not a causal claim, and once editors start acting on it there's a feedback-loop risk worth watching." |
| 4:30–5:00 | Close | Link to the deployed paper and repo. "Everything on the page traces back to a committed metrics file — rerun the notebook, get the same numbers." |

### Social-post cut

> A transparent "stale + visible" content-refresh rule sounds reasonable — until you check it by hand and find it only ever fires on 17 of 30,000 pages. A simple model trained on the same pre-decision signals ranks declining pages at Precision@50 = 0.70 vs. the rule's tie-aware 0.545, measured on clients it never saw during training. Method: Logistic Regression, evaluated on a client-holdout split so no client's pages leak between train and test. Full write-up + notebooks: https://github.com/SodiqAbdulwaris/flyrank-internship-ml
>
> [chart: `docs/assets/precision_comparison.svg`]

### Employer-facing summary (3 sentences)

> I built a decision-support ranking model that tells SEO editors which content page to review first, with a reason code attached to every recommendation. It's trained and validated on FlyRank's real anonymized search-performance data — 30,000 pages across 32 clients, trailing 90-day Google Search Console and GA4 metrics. It beats a transparent tie-aware rule baseline by 15.5 points of precision on a held-out set of clients it never saw during training (0.70 vs. 0.545), while a documented leakage-attack test and a random-vs-grouped split comparison both confirm the number is honest, not inflated.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
